# Calculate effective coverage by quintile and scenario

Also by age, sex, and pregnancy status (though coverage will not vary by pregnancy status due to a lack of data).

This is similar to what the pregnancy simulation does at the individual level, but using groups instead.
It can be shared between multiplication models that do not incorporate individual heterogeneity.

In [1]:
import pandas as pd

In [2]:
location = "india"
vehicle = "rice"
fortificant = "iron"

In [3]:
# Parameters
location = "india"
fortificant = "iron"
vehicle = "rice"


In [4]:
results_dir = f"../results/{fortificant}/{vehicle}"

In [5]:
full_coverage_probability = pd.read_csv(
    f"{results_dir}/baseline_fortification/full_coverage/{location}.csv"
)
full_coverage_probability = full_coverage_probability.set_index(
    [c for c in full_coverage_probability.columns if c != "value"]
).value
full_coverage_probability

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.156303
                            2                rice            0.173319
                            3                rice            0.172177
                            4                rice            0.123566
                            5                rice            0.062838
        5          15       1                rice            0.177188
                            2                rice            0.198127
                            3                rice            0.182862
                            4                rice            0.146883
                            5                rice            0.077928
        15         30       1                rice            0.174926
                            2                rice            0.205154
                            3                rice            0.183202
                            4   

In [6]:
any_coverage_probability = pd.read_csv(
    f"{results_dir}/baseline_fortification/any_coverage/{location}.csv"
)
any_coverage_probability = any_coverage_probability.set_index(
    [c for c in any_coverage_probability.columns if c != "value"]
).value
any_coverage_probability

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.560845
                            2                rice            0.535335
                            3                rice            0.492510
                            4                rice            0.454610
                            5                rice            0.293520
        5          15       1                rice            0.713577
                            2                rice            0.671205
                            3                rice            0.624088
                            4                rice            0.568540
                            5                rice            0.360006
        15         30       1                rice            0.642256
                            2                rice            0.612181
                            3                rice            0.545258
                            4   

In [7]:
partial_coverage_mean = pd.read_csv(
    f"{results_dir}/baseline_fortification/partial_coverage_amount/mean/{location}.csv"
)
partial_coverage_mean = partial_coverage_mean.set_index(
    [c for c in partial_coverage_mean.columns if c != "value"]
).value
partial_coverage_mean

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.524382
                            2                rice            0.552721
                            3                rice            0.551125
                            4                rice            0.530210
                            5                rice            0.471885
        5          15       1                rice            0.566689
                            2                rice            0.565275
                            3                rice            0.560457
                            4                rice            0.535734
                            5                rice            0.477592
        15         30       1                rice            0.535427
                            2                rice            0.563819
                            3                rice            0.558023
                            4   

In [8]:
current_coverage = (
    full_coverage_probability
    + (any_coverage_probability - full_coverage_probability) * partial_coverage_mean
)
current_coverage

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.368437
                            2                rice            0.373413
                            3                rice            0.348721
                            4                rice            0.299089
                            5                rice            0.171693
        5          15       1                rice            0.481154
                            2                rice            0.465546
                            3                rice            0.430151
                            4                rice            0.372779
                            5                rice            0.212646
        15         30       1                rice            0.425147
                            2                rice            0.434643
                            3                rice            0.385238
                            4   

In [9]:
scenarios = {
    "india": ["intervention"],
    "nigeria": ["intervention"],
    "ethiopia": ["intervention_25_nrv", "intervention_100_nrv"],
}[location]

In [10]:
fortifiability = pd.read_csv(
    f"../results/{vehicle}/vehicle_consumption/fortifiability/{location}.csv"
)
fortifiability = fortifiability.set_index(
    [c for c in fortifiability.columns if c != "value"]
).value
fortifiability

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.750637
                            2                rice            0.765043
                            3                rice            0.758215
                            4                rice            0.726727
                            5                rice            0.648110
        5          15       1                rice            0.809508
                            2                rice            0.806555
                            3                rice            0.787085
                            4                rice            0.751070
                            5                rice            0.659892
        15         30       1                rice            0.765629
                            2                rice            0.783191
                            3                rice            0.761378
                            4   

In [11]:
import pathlib

for scenario in scenarios:
    intervention_coverage = pd.read_csv(
        f"{results_dir}/{scenario}/intervention_fortification/any_coverage/{location}.csv"
    )
    intervention_coverage = intervention_coverage.set_index(
        [c for c in intervention_coverage.columns if c != "value"]
    ).value
    target_coverage = intervention_coverage * fortifiability
    display(target_coverage)
    assert (target_coverage > current_coverage.reindex_like(target_coverage)).all()
    # Not all coverage is effective -- this is as a proportion of coverage!
    effectiveness = pd.read_csv(
        f"{results_dir}/{scenario}/intervention_fortification/effectiveness/{location}.csv"
    )
    effectiveness = effectiveness.set_index(
        [c for c in effectiveness.columns if c != "value"]
    ).value
    effective_intervention_coverage = target_coverage * effectiveness
    path = f"{results_dir}/{scenario}/intervention_fortification/effective_coverage/{location}.csv"
    pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
    effective_intervention_coverage.reset_index().to_csv(path, index=False)

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.600510
                            2                rice            0.612034
                            3                rice            0.606572
                            4                rice            0.581381
                            5                rice            0.518488
        5          15       1                rice            0.647606
                            2                rice            0.645244
                            3                rice            0.629668
                            4                rice            0.600856
                            5                rice            0.527913
        15         30       1                rice            0.612503
                            2                rice            0.626553
                            3                rice            0.609102
                            4   

In [12]:
# Not all coverage is effective -- this is as a proportion of coverage!
effectiveness = pd.read_csv(
    f"{results_dir}/baseline_fortification/effectiveness/{location}.csv"
)
effectiveness = effectiveness.set_index(
    [c for c in effectiveness.columns if c != "value"]
).value

In [13]:
effective_baseline_coverage = current_coverage * effectiveness
effective_baseline_coverage

wealth_quintile  vehicle_name  sex     age_start  age_end
1                rice          Female  0          5          0.294750
                                       5          15         0.384923
                                       15         30         0.340118
                                       30         50         0.367724
                                       50         125        0.384701
                               Male    0          5          0.301125
                                       5          15         0.388070
                                       15         30         0.361587
                                       30         50         0.354325
                                       50         125        0.379405
2                rice          Female  0          5          0.298731
                                       5          15         0.372437
                                       15         30         0.347715
                                

In [14]:
path = f"{results_dir}/baseline_fortification/effective_coverage/{location}.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
effective_baseline_coverage.reset_index().to_csv(path, index=False)